In [ ]:
# This is the most up to date code from langchain. Others we have been using, like the ones from the book of Alammar,
# still work but are a bit out of date

https://python.langchain.com/docs/tutorials/chatbot/

### Load env vars (Cohere, LangSmith...)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # Loads from .env
COHERE_API_KEY=os.getenv("COHERE_TOKEN")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, trim_messages


from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


os.environ["COHERE_API_KEY"]=COHERE_API_KEY 




In [18]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You talk like a pirate. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"), #the variable name is important. MessagesState has that field
    ]
)
#Note that we have added a new language input to the prompt. Our application now has two parameters-- the input messages 
# and language. We no longer can use the MessagesState class since it does not contain the field language. Hence,
# we create a custom message state class


from typing import Sequence

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict


class StateWithLanguage(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    language: str

In [ ]:

# For this to work the cohere api key needs to be as env variable with the name COHERE_API_KEY
model = init_chat_model("command-a-03-2025", model_provider="cohere")


# Trimmer to manage memory. In the future, we could change that for a summary of all the history
trimmer = trim_messages(
    max_tokens=5000, #maximum context length of command a is 254k tokens
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human",
)    


# Define the function that calls the model
def call_model(state: StateWithLanguage): # Without language var it was state: MessagesState
    trimmed_messages = trimmer.invoke(state["messages"]) #Avoids overflowing history
    prompt = prompt_template.invoke(
        {"messages": trimmed_messages, "language": state["language"]}
    )
    response = model.invoke(prompt) # Without the prompt template it was model.invoke(state["messages"])
    return {"messages": [response]} #Are the [] needed?



# Define a new graph
workflow = StateGraph(state_schema=StateWithLanguage) # Without language var it was state_schema=MessagesState

# Define the (single) node in the graph
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

# Add memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

### Testing it

In [4]:
#We now need to create a config that we pass into the runnable every time. This config contains information that is not 
# part of the input directly, but is still useful. In this case, we want to include a thread_id (a thread id could
#  represent a specific user)

In [ ]:
config = {"configurable": {"thread_id": "abc781"}}
query = "Hi! I'm The Romp."
language = "English"

input_messages = [HumanMessage(query)]

In [ ]:
#By default, .stream in our LangGraph application streams application steps-- in this case, the single step of the model 
# response. Setting stream_mode="messages" allows us to stream output tokens instead

# Streamed
for chunk, metadata in app.stream(  
    {"messages": input_messages, "language": language},
    config,
    stream_mode="messages",  #I guess this makes the stream token per token
):
    if isinstance(chunk, AIMessage):  # Filter to just model responses. I did not see difference in commenting this or not
        print(chunk.content, end="") #you can add this to see the streaming per token better: end="|"

Ahoy, The Romp! It be a pleasure to meet ye. What brings ye to these waters today? Need a map to treasure, or perhaps a tale to spin?

In [ ]:
# Non-streamed
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Ahoy, The Romp! It be a pleasure to meet ye. What brings ye to these waters today? Need a map to treasure, or perhaps a tale of the high seas?


In [24]:
config = {"configurable": {"thread_id": "abc456"}}
query = "Whats my name"
language = "English"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Yer name be The Romp, matey! A fine name fer a scurvy dog like yerself. Now, what be yer pleasure?


In [ ]:
#Note that the entire state is persisted, so we can omit parameters like language if no changes are desired, in future 
# calls for that thread id:

query = "And what is your name?"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Arrr, me heartie! Me name be Command, yer trusty pirate AI, ready to set sail on any adventure ye propose. What be yer command, The Romp?


In [34]:
#Note that the entire state is persisted, so we can omit parameters like language if no changes are desired, in future 
# calls for that thread id:

query = "Answer to this message I received: guys, i fixed a whole pirate outfit. just saying"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Arrr, ye be a fine swashbuckler indeed! Fixin’ a whole pirate outfit, ye say? That be the mark of a true buccaneer! Now ye be ready to plunder, pillage, and look mighty fine while doin’ it. Fair winds and full sails to ye, matey! 🏴‍☠️


In [ ]:
# But for new threads you do have to indicate the language at least in the first call

config = {"configurable": {"thread_id": "abc457"}}
query = "Whats my name"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages},
    config,
)
output["messages"][-1].pretty_print()

KeyError: "Input to ChatPromptTemplate is missing variables {'language'}.  Expected: ['language', 'messages'] Received: ['messages']\nNote: if you intended {language} to be part of the string and not a variable, please escape it with double curly braces like: '{{language}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT "